# Module 6 — Python Client Architecture

Companion notebook to [`docs/modules/06_python_client_architecture.md`](../docs/modules/06_python_client_architecture.md).

This is the module where the canonical `LLMRuntime` abstraction gets built — the one every
later module depends on. Unlike every prior module, this notebook does **not** need to
honest-skip its core demonstrations: `FakeRuntime` and `httpx.MockTransport`-backed adapters
let every part of the abstraction be exercised for real, right here, without a live model
runtime. Jupyter supports top-level `await`, so every cell below calls the async API directly.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_01"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. FakeRuntime — the abstraction, exercised directly

In [2]:
from local_ai_core.runtimes.fake import FakeRuntime
from local_ai_core.runtimes.types import LLMRequest

fake = FakeRuntime(responses={"demo-model": "The KV cache stores keys and values per token per layer."})
response = await fake.generate(LLMRequest(model="demo-model", prompt="What is a KV cache?"))
print(f"Text: {response.text}")
print(f"Prompt tokens: {response.prompt_tokens}, completion tokens: {response.completion_tokens}")
print(f"Latency: {response.latency_ms:.2f}ms")

print("\nStreaming the same request:")
async for chunk in fake.stream(LLMRequest(model="demo-model", prompt="What is a KV cache?")):
    print(repr(chunk), end=" ")

Text: The KV cache stores keys and values per token per layer.
Prompt tokens: 5, completion tokens: 11
Latency: 0.00ms

Streaming the same request:
'The ' 'KV ' 'cache ' 'stores ' 'keys ' 'and ' 'values ' 'per ' 'token ' 'per ' 'layer. ' 

## 2. Retries: transient errors retry, validation errors don't

`FakeRuntime(fail_first_n_calls=2)` fails predictably so we can prove `with_retries` actually
retries transient errors — and then prove it does NOT retry a deterministic validation
failure (theory doc Lab 5).

In [3]:
from local_ai_core.runtimes.base import with_retries
from local_ai_core.runtimes.errors import SchemaValidationError

flaky = FakeRuntime(fail_first_n_calls=2, default_response="succeeded after retries")

async def call():
    return await flaky.generate(LLMRequest(model="m", prompt="p"))

result = await with_retries(call, max_attempts=5)
print(f"Result: {result.text!r} after {flaky.call_count} calls (2 failures + 1 success)")

not_retried = FakeRuntime(fail_with=SchemaValidationError("bad output shape"))

async def call_bad():
    return await not_retried.generate(LLMRequest(model="m", prompt="p"))

try:
    await with_retries(call_bad, max_attempts=5)
except SchemaValidationError:
    print(f"\nSchemaValidationError propagated immediately after {not_retried.call_count} call (no retries) - correct.")

Result: 'succeeded after retries' after 3 calls (2 failures + 1 success)

SchemaValidationError propagated immediately after 1 call (no retries) - correct.


## 3. OllamaRuntime against a mocked transport — real adapter code, no server needed

`httpx.MockTransport` exercises the adapter's real request-building, response-parsing, and
error-mapping code. See the theory doc's "Testing strategy" for what this does and does not
prove.

In [4]:
import httpx

from local_ai_core.runtimes.errors import RuntimeUnavailable
from local_ai_core.runtimes.ollama import OllamaRuntime

def mock_ollama_handler(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json={
        "response": "Grouped-query attention shares key/value projections across groups of query heads.",
        "done": True, "prompt_eval_count": 6, "eval_count": 12,
    })

mocked_ollama = OllamaRuntime(client=httpx.AsyncClient(transport=httpx.MockTransport(mock_ollama_handler)))
response = await mocked_ollama.generate(LLMRequest(model="qwen2.5:7b", prompt="What is GQA?"))
print(f"Text: {response.text}")
print(f"Tokens: {response.prompt_tokens} prompt / {response.completion_tokens} completion")

# Error normalization: a connection failure becomes a typed LLMError, never a raw httpx exception.
def refusing_handler(request: httpx.Request) -> httpx.Response:
    raise httpx.ConnectError("connection refused", request=request)

unreachable = OllamaRuntime(client=httpx.AsyncClient(transport=httpx.MockTransport(refusing_handler)))
try:
    await unreachable.generate(LLMRequest(model="m", prompt="p"))
except RuntimeUnavailable as exc:
    print(f"\nCorrectly raised RuntimeUnavailable (not a raw httpx.ConnectError): {exc}")

Text: Grouped-query attention shares key/value projections across groups of query heads.
Tokens: 6 prompt / 12 completion

Correctly raised RuntimeUnavailable (not a raw httpx.ConnectError): Could not connect to Ollama: connection refused


## 4. MLXRuntime with injected fakes — proving the async bridge over a sync generator works

In [5]:
from dataclasses import dataclass
from local_ai_core.runtimes.mlx import MLXRuntime

class DemoTokenizer:
    def apply_chat_template(self, messages, tokenize, add_generation_prompt):
        return "".join(f"<{m['role']}>{m['content']}" for m in messages) + "<assistant>"
    def encode(self, text):
        return list(range(len(text.split())))

@dataclass(frozen=True)
class DemoStreamItem:
    text: str

def demo_load_fn(model_id):
    return (f"mlx-model:{model_id}", DemoTokenizer())

def demo_generate_fn(model, tokenizer, *, prompt, max_tokens):
    return "MLX runs natively on Apple Silicon unified memory."

def demo_stream_generate_fn(model, tokenizer, *, prompt, max_tokens):
    for word in ["MLX ", "streams ", "token ", "by ", "token."]:
        yield DemoStreamItem(text=word)

mlx_runtime = MLXRuntime(load_fn=demo_load_fn, generate_fn=demo_generate_fn, stream_generate_fn=demo_stream_generate_fn)
response = await mlx_runtime.generate(LLMRequest(model="demo-mlx-model", prompt="Tell me about MLX."))
print(f"Text: {response.text}")

print("\nStreaming (genuinely incremental, bridged from a sync generator via a background thread):")
async for chunk in mlx_runtime.stream(LLMRequest(model="demo-mlx-model", prompt="Tell me about MLX.")):
    print(repr(chunk), end=" ")

Text: MLX runs natively on Apple Silicon unified memory.

Streaming (genuinely incremental, bridged from a sync generator via a background thread):
'MLX ' 'streams ' 'token ' 'by ' 'token.' 

## 5. The abstraction actually abstracts: same request, four adapters, same response shape

In [6]:
runtimes = {"fake": fake, "ollama (mocked)": mocked_ollama, "mlx (fake-injected)": mlx_runtime}

print(f"{'Adapter':22s} {'text is str':>12s} {'model matches':>15s} {'has latency_ms':>16s}")
for name, rt in runtimes.items():
    req = LLMRequest(model="contract-demo-model", prompt="hi")
    resp = await rt.generate(req)
    print(f"{name:22s} {str(isinstance(resp.text, str)):>12s} {str(resp.model == req.model):>15s} {str(resp.latency_ms is not None):>16s}")

print("\nSee packages/local_ai_core/runtimes/tests/test_runtime_contract.py for the full,")
print("pytest-enforced version of this proof (22 passing, 2 correctly-skipped assertions).")

Adapter                 text is str   model matches   has latency_ms
fake                           True            True             True
ollama (mocked)                True            True             True
mlx (fake-injected)            True            True             True

See packages/local_ai_core/runtimes/tests/test_runtime_contract.py for the full,
pytest-enforced version of this proof (22 passing, 2 correctly-skipped assertions).


## 6. Real Ollama, if available

Expected to skip on this machine (no local model runtime installed by design).

In [7]:
from ollama_probe import is_ollama_available

if is_ollama_available():
    real_ollama = OllamaRuntime()
    real_response = await real_ollama.generate(LLMRequest(model="qwen2.5:1.5b", prompt="Say hello in five words."))
    print(f"Real response: {real_response.text}")
    await real_ollama.aclose()
else:
    print("SKIPPED: Ollama is not reachable at http://localhost:11434.")
    print("This machine deliberately has no local model runtime installed (see PROGRESS.md).")
    print("On the resourced 32GB Mac: OllamaRuntime() with no arguments points at localhost:11434 by default.")

SKIPPED: Ollama is not reachable at http://localhost:11434.
This machine deliberately has no local model runtime installed (see PROGRESS.md).
On the resourced 32GB Mac: OllamaRuntime() with no arguments points at localhost:11434 by default.


## 7. Run the full test suite

```bash
uv run pytest packages/local_ai_core/runtimes -q
```

165 tests (2 correctly skipped), all passing without any model runtime — this module's whole
point is that the abstraction is fully verifiable without one.